# HuggingFace Transformers - The Basics
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/05_NLP_Embeddings/huggingface_transformers_basics.ipynb)

The `transformers` library gives one-line access to thousands of pre-trained models. The high-level `pipeline()` API covers the most common NLP tasks; below it sit tokenizers and models you control directly.

Runs free on Google Colab (models ~50-500 MB download on first use).

In [ ]:
!pip install -q transformers

## 1. The pipeline() API

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I love how simple this API is!"))
print(classifier(["Worst support ever.", "Delivery arrived early."]))

In [ ]:
ner = pipeline("ner", grouped_entities=True)
ner("Ajit works at Quantsmind in Pune and joined in 2024.")

In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6")
article = ("Large language models are neural networks trained on massive text corpora "
           "to predict the next token. This simple objective, combined with scale, "
           "produces emergent abilities such as in-context learning, reasoning and "
           "tool use. Companies now fine-tune these models on domain data and align "
           "them with human feedback.")
summarizer(article, max_length=40, min_length=15)

In [ ]:
zero_shot = pipeline("zero-shot-classification")
zero_shot("Breakfast at 8am, standup at 9:30, deploy at 11.",
          candidate_labels=["work", "food", "travel", "sports"])

## 2. Under the hood: tokenizer + model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

ckpt = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt)

enc = tok("Transformers read text as numbers!", return_tensors="pt")
print("tokens :", tok.convert_ids_to_tokens(enc["input_ids"][0]))
with torch.no_grad():
    logits = model(**enc).logits
probs = torch.softmax(logits, dim=-1)
print("probs  :", dict(zip(model.config.id2label.values(), probs[0].tolist())))

## Cheat sheet
| Task | pipeline name | Typical output |
|---|---|---|
| sentiment | `"sentiment-analysis"` | label + score |
| entities | `"ner"` | spans |
| summary | `"summarization"` | text |
| classify w/o training | `"zero-shot-classification"` | ranked labels |
| answers from context | `"question-answering"` | span |
| generate text | `"text-generation"` | continuation |

Model Hub tip: filter by task + size (`distil*`, `mini`, `small`) for free-tier GPUs.